In [ ]:
import numpy as np
import torch
torch.set_default_dtype(torch.float32)
import gpytorch

import gc

dtype = torch.float32
device = 'cuda:0'

In [ ]:
def free():
    gc.collect()
    torch.cuda.empty_cache()

# Training

In [ ]:
n_samples = 1000

idx = np.random.choice(range(5848), size=n_samples, replace=False)
X_train = torch.tensor(np.load('data/X_train.npy')[idx], dtype=dtype, device=device)
y_train = torch.tensor(np.load('data/y_train.npy')[idx], dtype=dtype, device=device)
y_mean = y_train.mean()
y_std = y_train.std()
y_train = (y_train - y_mean) / y_std

In [ ]:
mean_in = y_train[X_train[:,-1] == 1].mean()
mean_out = y_train[X_train[:,-1] == 0].mean()

In [ ]:
class GroupMeans(gpytorch.means.Mean):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return mean_in * x[:,2] + mean_out * (1 - x[:,2])

In [ ]:
try: del model
except: pass
    
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = GroupMeans()
        self.covar_module = gpytorch.kernels.RBFKernel(
            lengthscale_prior=gpytorch.priors.GammaPrior(1, 1)
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
likelihood.noise = y_train.var()
model = ExactGPModel(X_train, y_train, likelihood).to(device)

In [ ]:
# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.NAdam(model.parameters(), lr=0.01)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

training_iter = 1000
for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(X_train)
    # Calc loss and backprop gradients
    loss = -mll(output, y_train)
    loss.backward()
    if i % 100 == 0:
        print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
            # model.covar_module.base_kernel.lengthscale.item(),
            # model.covar_module.outputscale.item(),
            model.covar_module.lengthscale.item(),
            model.likelihood.noise.item()
        ))
    optimizer.step()

In [ ]:
X_test = torch.tensor(np.load('data/X_test.npy'), dtype=dtype, device=device)

# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

# Test points are regularly spaced along [0,1]
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    y_hat = likelihood(model(X_test))

# Plotting

In [ ]:
from wifiplotting import *
import dill

with open('data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)
wlon_train, wlat_train = np.load('data/world_train.npy').T
wlon_test, wlat_test = np.load('data/world_test.npy').T

In [ ]:
with torch.no_grad():
    vmax = y_train.max()
    vmin = y_train.min()
    
    base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)
    
    sc = base_ax.scatter(wlon_test, wlat_test, c=y_hat.mean.cpu().numpy(), cmap="RdYlGn", vmin=vmin, vmax=vmax)
    
    base_ax.scatter(wlon_train[idx], wlat_train[idx], c=y_train.cpu().numpy(), cmap="RdYlGn",
                    vmin=vmin, vmax=vmax, alpha=0.4)
    plt.ticklabel_format(style='plain', axis='both', useOffset=False)
    
    plt.colorbar(sc)
    plt.tight_layout()